# Delay-transform / in-painting scratch (from rfi_dev_v2.1)

Split out of `rfi_dev_v2.1.ipynb` on 2026-09-21 when that notebook was made the
canonical RFI review interface. These cells are exploratory: they in-paint the
flagged residual with model noise and look at the delay transform. They depend on
`air`, `result`, `flagged_residual` and `config` from rfi_dev_v2.1 sections 1-3,
so run that notebook first, or paste its setup here.


In [ ]:
from hera_filters.dspec import dpss_operator, fit_solution_matrix
freqs = air.freqs_mhz * 1e6
dt = air.meta['integration_time'][0]
B = 250e6 / air.meta['nchan'][0]
noise_lev = 2 * result.model / np.sqrt(2 * B * dt)
noise = np.random.normal(size=noise_lev.shape) * noise_lev
inpainted = np.where(result.mask, flagged_residual, noise)
A = dpss_operator(freqs, filter_centers=[500e-9, -500e-9], filter_half_widths=[300e-9, 300e-9], eigenval_cutoff=[1e-12, 1e-12])[0]
#A = dpss_operator(freqs, filter_centers=[0], filter_half_widths=[200e-9], eigenval_cutoff=[1e-12])[0].real
print(A.shape, inpainted.shape)
c = A.T @ inpainted.T
print(A.shape, c.shape)
mdl = np.real(A @ c).T

In [ ]:
print(inpainted)

In [ ]:
plt.figure()
plt.semilogy(air.freqs_mhz, np.mean(air.data, axis=0))
plt.semilogy(air.freqs_mhz, np.mean(result.model, axis=0))
plt.semilogy(air.freqs_mhz, np.mean(noise_lev, axis=0))
#plt.plot(air.freqs_mhz, np.sqrt(np.nansum(np.abs(flagged_residual)**2, axis=0) / np.sum(result.mask, axis=0).clip(1)))
plt.plot(air.freqs_mhz, np.abs(flagged_residual[0]))
plt.plot(air.freqs_mhz, np.abs(inpainted[0]))
plt.plot(air.freqs_mhz, np.abs(mdl[0]))
#plt.plot(air.freqs_mhz, np.abs(filt_flag_res))

In [ ]:
dlys = np.fft.fftfreq(air.meta['nchan'][0], 250e6 / air.meta['nchan'][0]) * 1e9
dlys[50], dlys[200]

In [ ]:
fig, axes = plt.subplots(ncols=2)
axes[0].imshow(np.abs(np.fft.fft(np.nan_to_num(inpainted[:100], nan=0), axis=1)), aspect='auto', interpolation='nearest', cmap='plasma')
axes[1].imshow(np.abs(np.fft.fft(np.nan_to_num(result.mask[:100] * mdl, nan=0), axis=1)), aspect='auto', interpolation='nearest', cmap='plasma')